# Game Enviornment

In [1]:
import random
from collections import deque
import math
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import json
import os

BOARD_SIZE = 4
ACTIONS = [0, 1, 2, 3]  # up,right, down, left

def add_tile(board):
    empty = list(zip(*np.where(board == 0)))
    if not empty:   # no empty cells
        return board
    y, x = random.choice(empty)
    board[y][x] = 1 if random.random() < 0.9 else 2
    return board

def move_right(board):
    new_board = np.zeros_like(board)
    reward = 0
    for row in range(BOARD_SIZE):
        tiles = board[row][board[row] != 0] # collect non-zero tiles
        merged = []
        skip = False
        for i in range(len(tiles)):
            if skip:
                skip = False
                continue
            if i + 1 < len(tiles) and tiles[i] == tiles[i+1]:
                merged.append(tiles[i] + 1)
                reward += 2 ** (tiles[i] + 1)  # calculate reward
                skip = True
            else:
                merged.append(tiles[i])
        new_board[row][:len(merged)] = merged
    return new_board, reward

def move(board, direction): 
    if direction == 0:  # up
        board = np.rot90(board, 1)
        new_board, reward = move_right(board)   #reuse this func to death bc im lazy lmao
        new_board = np.rot90(new_board, -1)
    elif direction == 2:  # down
        board = np.rot90(board, -1)
        new_board, reward = move_right(board)
        new_board = np.rot90(new_board)
    elif direction == 3:  # left
        new_board, reward = move_right(board)
    elif direction == 1:  # right
        board = np.fliplr(board)
        new_board, reward = move_right(board)
        new_board = np.fliplr(new_board)
    else:
        raise ValueError("Invalid direction")
    return new_board, reward

def is_game_over(board):
    for a in ACTIONS:
        new_board, _ = move(board, a)
        if not np.array_equal(new_board, board):
            return False
    return True

class Game2048Env:
    def reset(self):
        self.board = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=int)
        self.board = add_tile(add_tile(self.board))
        return self.get_state()

    def step(self, action):
        # old_max_tile = np.max(self.board)
        old_board = self.board.copy()
        self.board, reward = move(self.board, action)
        changed = not np.array_equal(self.board, old_board)
        if changed: # only add a tile if the board changed
            self.board = add_tile(self.board)
        # new_max_tile = np.max(self.board)
        # reward = (new_max_tile > old_max_tile)  # reward for increasing max tile, small reward for merging
        done = is_game_over(self.board)
        return self.get_state(), reward, done

    def get_state(self):
        board = self.board.copy()
        board = np.where(board > 0, board, 0)
        board = board.astype(np.float32)
        board = board.reshape(1, 1, 4, 4)
        return board

# Neural Network Implementation

In [ ]:
import torch
from torch import nn
from torch import optim

output_dim = 4

class CNN(nn.Module):  # Use CNN because input is image-like (4x4 grid)
    def __init__(self, output_dim=4, dropout_prob=0.3):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=2, stride=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=2, stride=1)
        self.fc1 = nn.Linear(128 * 2 * 2, 128)
        self.dropout = nn.Dropout(dropout_prob)  # add dropout bc i dont have many games
        self.fc2 = nn.Linear(128, output_dim)
    
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x) 
        return self.fc2(x)

model = CNN(output_dim=output_dim)
print(model)    

# Data Processing Functions

In [3]:
import os
import shutil
import json
import numpy as np

def rotate_cells(state):
    grid = np.array(state).reshape(4, 4)
    rotated_grid = np.rot90(grid, k=2)  #rotate 180deg
    return rotated_grid.flatten().tolist()

def rotate_file(input_path, output_path, small):
    with open(input_path, 'r') as infile, open(output_path, 'w') as outfile:
        for line in infile:
            json_obj = json.loads(line)
            max_value = max(json_obj['state'])
            if (max_value < 512 and not small):
                continue
            json_obj['state'] = rotate_cells(json_obj['state'])
            json_obj['action'] = (json_obj['action'] + 1) % 4
            outfile.write(json.dumps(json_obj) + '\n')

def mirror_cells(state):
    grid = np.array(state).reshape(4, 4)
    mirrored_grid = np.fliplr(grid)  # mirror horizontally
    return mirrored_grid.flatten().tolist()

def mirror_file(input_path, output_path, small):
    with open(input_path, 'r') as infile, open(output_path, 'w') as outfile:
        for line in infile:
            mirdirs = [0, 3, 2, 1]
            json_obj = json.loads(line)
            max_value = max(json_obj['state'])
            if (max_value < 512 and not small):
                continue
            json_obj['state'] = mirror_cells(json_obj['state'])
            json_obj['action'] = mirdirs[json_obj['action']]
            outfile.write(json.dumps(json_obj) + '\n')

evil = 2

def pick_file(input_path, output_path):
    with open(input_path, 'r') as infile, open(output_path, 'w') as outfile:
        for line in infile:
            json_obj = json.loads(line)
            if (json_obj['action'] != evil):
                outfile.write(json.dumps(json_obj) + '\n')

def clear_files():
    base_dir = os.path.expanduser("data")
    preserve_dir = os.path.join(base_dir, "raw")
    for item in os.listdir(base_dir):
        path = os.path.join(base_dir, item)

        # Skip the preserved directory
        if os.path.abspath(path) == os.path.abspath(preserve_dir):
            continue
        if os.path.isfile(path) or os.path.islink(path):
            os.remove(path)
        if os.path.isdir(path):
            shutil.rmtree(path)

# Training

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader

learning_rate = 0.001
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
num_epochs = 50
save_dir = "experimental_models"
os.makedirs(save_dir, exist_ok=True)
descriptions = ["8+small+evil", "8+small", "8+evil", "8rots", "4+small+evil", "4+small", "4+evil", "4rots"]
iterations = len(descriptions)

avg_scores = np.zeros((iterations,num_epochs))
tile_distribution = np.zeros((iterations, num_epochs, 17), dtype=int)
best_epoch = np.zeros(iterations, dtype=int)


def train(descr, index):
    data_dir = "data/"
    X = []
    Y = []

    for subdir in os.listdir(data_dir):
        subdir_path = os.path.join(data_dir, subdir)
        for file_name in os.listdir(subdir_path):
            file_path = os.path.join(subdir_path, file_name)
            with open(file_path, "r") as file:
                # print(f"file: {file_path}")
                for line in file:
                    data = json.loads(line.strip())
                    if "state" in data and "action" in data:
                        X.append(data["state"])
                        Y.append(data["action"])
    X = np.array(X)
    X[X > 0] = np.log2(X[X > 0])    #replace with log2 for simplicity

    X_train = np.array(X)
    y_train = np.array(Y)   #overwrite with full dataset for training

    #convert data to torch tensors
    class Data(Dataset):
        def __init__(self, X, y):   #reshape to fit CNN input, -1 to auto infer batch size, 1 for single channel
            self.X = torch.from_numpy(X.astype(np.float32)).reshape(-1, 1, 4, 4)
            self.y = torch.from_numpy(y.astype(np.float32))
            self.len = self.X.shape[0]
        
        def __getitem__(self, index):
            return self.X[index], self.y[index]
    
        def __len__(self):
            return self.len
    
    batch_size = 64
    train_data = Data(X_train, y_train)
    train_dataloader = DataLoader(dataset=train_data, batch_size=batch_size, shuffle=True)
    

    for epoch in tqdm(range(num_epochs)):
        epoch_loss = 0.0
        batch_count = 0
        model.train()
        for X, y in train_dataloader:
            optimizer.zero_grad()
            pred = model(X)
            loss = loss_fn(pred, y.long())
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            batch_count += 1
        model.eval()
        for i in range(100):
            env = Game2048Env()
            state = env.reset()
            done = False
            score = 0
            while not done:
                board = state.copy().flatten(order='F')
                state_tensor = torch.tensor(board, dtype=torch.float32).reshape(-1, 1, 4, 4)
                logits = model(state_tensor)
                ranked_actions = torch.argsort(logits, dim=1, descending=True)[0]
                original_board = env.board.copy()
                final_action = None
                for action in ranked_actions:
                    test_board, _ = move(original_board.copy(), action.item())
                    if not np.array_equal(test_board, original_board):
                        final_action = action.item()
                        break
                state, reward, done = env.step(final_action)
                score += reward
            avg_scores[index][epoch] += score/100
            tile_distribution[index][epoch][env.board.max()] += 1
        if (avg_scores[index][epoch] >= avg_scores[index][best_epoch[index]]):
            best_epoch[index] = epoch
            model_path = os.path.join(save_dir, f"{descr}.pth")
            torch.save(model.state_dict(), model_path)

# Test Iters

In [ ]:
from time import sleep
raw_dir = 'data/raw'
zero_dir = 'data/rot0'

for iter in tqdm(range(iterations)):
    clear_files()

    os.makedirs(zero_dir, exist_ok=True)
    if os.path.exists("data/.DS_Store"):
        os.remove("data/.DS_Store")
    if os.path.exists("data/raw/.DS_Store"):
        os.remove("data/raw/.DS_Store")

    for in_name in os.listdir(raw_dir):
        in_path = os.path.join(raw_dir, in_name)
        if os.path.isfile(in_path) and in_name.endswith('.jsonl'):
            output_file_name = f"rot0_{in_name}"
            output_path = os.path.join(zero_dir, output_file_name)
            if (iter % 2 == 1):
                pick_file(in_path, output_path)
                in_path = output_path
            for i in range(4):
                if (i % 2 == 1 and iter > iterations // 2): #for 4rots, ignore 90 and 270
                    continue
                if (i != 0):
                    out_dir = f"data/rot{i*90}"
                    os.makedirs(out_dir, exist_ok=True)
                    output_file_name = f"rot{i*90}_{in_name}"
                    output_path = os.path.join(out_dir, output_file_name)
                    rotate_file(in_path, output_path, (iter % 4 < 2))
                    # print(f"created {output_file_name}")
                    in_path = output_path

                out_dir = f"data/rot{i*90}-mir"
                os.makedirs(out_dir, exist_ok=True)
                output_file_name = f"rot{i*180}_mir_{in_name}"
                output_path = os.path.join(out_dir, output_file_name)
                mirror_file(in_path, output_path, (iter % 4 < 2))
                # print(f"created {output_file_name}")
    train(descriptions[iter], iter)